In [ ]:

import sqlite3

# Connect to SQLite (file-based to persist between cells)
conn = sqlite3.connect("users.db")
cursor = conn.cursor()

# Create table
cursor.execute('''
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT UNIQUE,
    password TEXT,
    email TEXT,
    secret TEXT
)
''')

conn.commit()
print("✅ Table created successfully")


In [ ]:

import pyotp

def register_user(username, password, email):
    # Generate a unique OTP secret
    secret = pyotp.random_base32()

    cursor.execute("INSERT OR REPLACE INTO users (username, password, email, secret) VALUES (?, ?, ?, ?)",
                   (username, password, email, secret))
    conn.commit()
    print(f"✅ User '{username}' registered with secret: {secret}")
    return secret

# Example user
secret = register_user("demo_user", "change_me", "demo@example.com")


In [ ]:

import qrcode
from IPython.display import Image, display

def generate_qr(username, secret):
    totp = pyotp.TOTP(secret)
    uri = totp.provisioning_uri(name=username, issuer_name="MyApp")
    qr = qrcode.make(uri)
    qr.save("otp_qr.png")
    display(Image(filename="otp_qr.png"))
    print("✅ Scan this QR in Google Authenticator")

generate_qr("demo_user", secret)


In [ ]:

def login(username, password, otp_code):
    cursor.execute("SELECT password, secret FROM users WHERE username=?", (username,))
    row = cursor.fetchone()
    if row:
        stored_password, secret = row
        if password == stored_password:
            totp = pyotp.TOTP(secret)
            if totp.verify(otp_code):
                print("✅ Login successful!")
                return True
            else:
                print("❌ Invalid OTP")
        else:
            print("❌ Invalid password")
    else:
        print("❌ User not found")
    return False

# Example: enter OTP from your Google Authenticator app
# otp_input = input("Enter OTP: ")
# login("testuser", "testpass", otp_input)


In [ ]:

import requests

def get_joke():
    url = "https://official-joke-api.appspot.com/random_joke"
    response = requests.get(url).json()
    joke = f"{response['setup']} - {response['punchline']}"
    print("😂 Joke:", joke)
    return joke

# Example
joke_text = get_joke()


In [ ]:

def generate_joke_qr(joke_text):
    qr = qrcode.make(joke_text)
    qr.save("joke_qr.png")
    display(Image(filename="joke_qr.png"))
    print("✅ QR Code generated for joke")

generate_joke_qr(joke_text)
